# Estimating the Treatment Effect

## Objective

The exploratory analysis showed that the treatment and control groups are well balanced across observed customer characteristics. This suggests that the marketing campaign closely resembles a randomized controlled experiment.

In this notebook, we estimate the causal impact of the email campaign by comparing customer outcomes between the treatment and control groups.

Specifically, we aim to answer the following questions:

- Did the campaign increase customer conversions?
- Did the campaign increase customer spending?
- Are the observed differences statistically significant?
- What is the magnitude of the treatment effect?

### Step 1 — Import Libraries

In [3]:
import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

### Step 2 — Load data

In [4]:
df = pd.read_csv("../data/raw/hillstrom.csv")

df["treatment"] = (
    df["segment"] != "No E-Mail"
).astype(int)

In [5]:
conversion = (
    df.groupby("treatment")["conversion"]
      .agg(
          Customers="count",
          Conversions="sum",
          Conversion_Rate="mean"
      )
)

conversion

,Customers,Conversions,Conversion_Rate
treatment,,,
0,21306,122,0.005726
1,42694,456,0.010681


### Estimate the ATE

The Average Treatment Effect (ATE) measures the average change in an outcome if every customer were exposed to the marketing campaign compared with if no customers were exposed.

Since this dataset approximates a randomized experiment, the ATE can be estimated by the difference in average outcomes between the treatment and control groups.

We begin by estimating the treatment effect on customer conversions.

In [6]:
control_rate = (
    conversion.loc[0, "Conversion_Rate"]
)

treatment_rate = (
    conversion.loc[1, "Conversion_Rate"]
)

ate = treatment_rate - control_rate

relative_lift = ate / control_rate

In [7]:
print(f"Control Conversion Rate   : {control_rate:.4%}")
print(f"Treatment Conversion Rate : {treatment_rate:.4%}")
print(f"ATE (Absolute Lift)       : {ate:.4%}")
print(f"Relative Lift             : {relative_lift:.2%}")

Control Conversion Rate   : 0.5726%
Treatment Conversion Rate : 1.0681%
ATE (Absolute Lift)       : 0.4955%
Relative Lift             : 86.53%


## Interpretation

The treatment group achieved a higher conversion rate than the control group.

The difference in conversion rates represents the **Average Treatment Effect (ATE)** because treatment assignment was approximately randomized.

From a business perspective, this quantity is also referred to as the **absolute conversion lift**, while the relative lift expresses the proportional improvement over the baseline conversion rate.

### Confidence Interval

In [8]:
count = np.array([
    conversion.loc[1, "Conversions"],
    conversion.loc[0, "Conversions"]
])

nobs = np.array([
    conversion.loc[1, "Customers"],
    conversion.loc[0, "Customers"]
])

ci_low, ci_high = confint_proportions_2indep(
    count1=count[0],
    nobs1=nobs[0],
    count2=count[1],
    nobs2=nobs[1],
    method="score"
)

print(f"95% CI: ({ci_low:.4%}, {ci_high:.4%})")

95% CI: (0.3517%, 0.6341%)


## Confidence Interval

The confidence interval provides a range of plausible values for the true treatment effect.

If the interval does not include zero, the observed treatment effect is statistically significant at the 5% significance level.

### Hypothesis Test

In [10]:
z_stat, p_value = proportions_ztest(
    count,
    nobs
)

print(f"Z-statistic : {z_stat:.3f}")
print(f"P-value     : {p_value:.12f}")

Z-statistic : 6.244
P-value     : 0.000000000427


# Conclusion

The objective of this notebook was to estimate the impact of the email marketing campaign using the randomized experiment.

The treatment group achieved a higher conversion rate than the control group, resulting in an estimated **Average Treatment Effect (ATE)** of approximately **0.50 percentage points**, corresponding to an **86.5% relative lift** in conversions.

Statistical inference showed that:

- The observed treatment effect is statistically significant.
- The 95% confidence interval excludes zero, indicating that the campaign had a positive impact on customer conversions.
- The hypothesis test produced a very small p-value, providing strong evidence against the null hypothesis of no treatment effect.

Because the treatment and control groups were well balanced in the previous exploratory analysis, this difference can reasonably be interpreted as the **causal effect** of the marketing campaign rather than the result of systematic differences between customers.

However, randomized experiments are not always feasible in real-world settings. Many organizations rely on observational data, where treatment assignment is influenced by customer characteristics such as purchase history, engagement, or demographics. In such cases, naïve comparisons between treated and untreated customers can produce biased estimates of treatment effects.

In the next notebook, we will simulate an observational marketing campaign by introducing selection bias into the treatment assignment process. This will create a realistic scenario where simple differences in outcomes are no longer valid estimates of causal effects, motivating the use of methods such as **Propensity Score Matching (PSM)** to recover unbiased treatment effect estimates.